In [2]:
import pandas as pd
import numpy as np
import joblib

### wczytanie danych
df = joblib.load('df_with_all_clusters.joblib')
embeddings_full = np.load('embeddings_full.npy')
print("Obiekty wczytane.")

### sprawdzenie że czy jest tyle samo embeddingów co wierszy w DataFrame
assert len(df) == len(embeddings_full), "Niezgodna liczba dokumentów i embeddingów!"

Obiekty wczytane.


In [4]:
### potrzebna lista unikalnych ID dla każdego dokumentu (najlepiej indeksy DataFrame)
### ChromaDB wymaga ID jako stringów!!!
doc_ids = [str(i) for i in df.index]

print(f"Przygotowano dane dla {len(doc_ids)} dokumentów.")

Przygotowano dane dla 850367 dokumentów.


In [6]:
### Inicjalizacja ChromaDB i dodanie danych

import chromadb
import time
from tqdm import tqdm


#### zarządzanie bazą danych w folderze 'chroma_db'
client = chromadb.PersistentClient(path="chroma_db")


#### nazwa kolekcji (odpowiednik tabeli w SQL)
collection_name = "covid_articles"


try:
    client.delete_collection(name=collection_name)
    print(f"Usunięto istniejącą kolekcję: {collection_name}")
except:
    pass


#### nowa kolekcja
print(f"Tworzenie nowej kolekcji: {collection_name}")
collection = client.create_collection(name=collection_name)


#### dodawanie danych do kolekcji w partiach
batch_size = 5000
num_batches = int(np.ceil(len(doc_ids) / batch_size))


print(f"\nDodawanie {len(doc_ids)} dokumentów do bazy w {num_batches} partiach...")
start_time = time.time()


for i in tqdm(range(num_batches)):
    start_idx = i * batch_size
    end_idx = min((i + 1) * batch_size, len(doc_ids))
    
    #### dane dla bieżącej partii
    batch_ids = doc_ids[start_idx:end_idx]
    #### ChromaDB oczekuje listy wektorów a nie tablicy NumPy
    batch_embeddings = embeddings_full[start_idx:end_idx].tolist() 
    
    #### dodanie partii do kolekcji
    collection.add(
        embeddings=batch_embeddings,
        ids=batch_ids
    )


end_time = time.time()
print(f"\nDane zostały dodane do bazy ChromaDB w {(end_time - start_time) / 60:.2f} minut.")

Tworzenie nowej kolekcji: covid_articles

Dodawanie 850367 dokumentów do bazy w 171 partiach...


100%|██████████| 171/171 [05:26<00:00,  1.91s/it]


Dane zostały dodane do bazy ChromaDB w 5.44 minut.
